In [1]:
import sys
print('Python:', sys.version)
print('Executable:', sys.executable)

# Install/upgrade into *this* notebook kernel environment (keep LangChain deps consistent)
%pip install -U langchain langchain-community langchain-text-splitters langchain-core langsmith langchain-google-genai

Python: 3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 13:17:27) [MSC v.1929 64 bit (AMD64)]
Executable: c:\Users\User\anaconda3\python.exe
Note: you may need to restart the kernel to use updated packages.


In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("Missing API key. Set GOOGLE_API_KEY or GEMINI_API_KEY (for example in a .env file).")

# Pick a model that actually exists for your API key.
# You can override this by setting GEMINI_MODEL in your environment.
requested_model = os.getenv("GEMINI_MODEL")

available_models: list[str] = []
if not requested_model:
    # Prefer the modern `google.genai` SDK (used internally by langchain-google-genai).
    try:
        from google import genai
        client = genai.Client(api_key=api_key)
        for m in client.models.list():
            name = getattr(m, "name", "") or ""
            methods = getattr(m, "supported_generation_methods", []) or []
            if name and "generateContent" in methods:
                available_models.append(name.removeprefix("models/"))
    except Exception:
        available_models = []

    # Fallback for older environments where only `google-generativeai` works.
    if not available_models:
        try:
            import google.generativeai as genai_legacy
            genai_legacy.configure(api_key=api_key)
            for m in genai_legacy.list_models():
                name = getattr(m, "name", "") or ""
                methods = getattr(m, "supported_generation_methods", []) or []
                if name and "generateContent" in methods:
                    available_models.append(name.removeprefix("models/"))
        except Exception:
            available_models = []

model = requested_model
if not model:
    if not available_models:
        raise ValueError(
            "No Gemini models were discovered for this API key. "
            "Set GEMINI_MODEL to a valid model name (from Google's ListModels)."
        )
    model = next((m for m in available_models if "flash" in m.lower()), available_models[0])

llm = ChatGoogleGenerativeAI(
    model=model,
    api_key=api_key,
 )

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Summarize the topic clearly."),
    ("human", "{topic}")
])

chain = prompt | llm

response = chain.invoke({"topic": "What is machine learning?"})
print(response.content)

C:\Users\User\AppData\Local\Temp\ipykernel_12632\2093900460.py:33: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai_legacy


**Machine learning (ML)** is a field of artificial intelligence that enables computers to **"learn" from data without being explicitly programmed** for every task.

Instead of a human programmer writing detailed instructions for every possible scenario, machine learning algorithms are fed large amounts of data. From this data, they identify patterns, relationships, and insights, and then use that "learned" knowledge to make predictions, classifications, or decisions on new, unseen data.

Here's a breakdown of the core idea:

1.  **Data is Key:** ML models are trained on vast datasets. For example, to recognize cats, you'd feed it thousands of images labeled "cat" and "not a cat."
2.  **Pattern Recognition:** The algorithm analyzes this training data to find recurring patterns, features, and correlations. It learns what characteristics define a "cat."
3.  **Model Creation:** The output of this learning process is a "model" – essentially, the learned rules or relationships.
4.  **Predict

In [3]:
! pip install langchain-community pypdf

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load PDF
loader = PyPDFLoader(r"C:\Users\User\Downloads\Note 11 - Linear Regression with Multiple Responses.pdf")
pages = loader.load()

print(f"Total pages: {len(pages)}")
print(pages[0].page_content[:500])  # preview first page

# Split into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)
chunks = splitter.split_documents(pages)
print(f"Total chunks: {len(chunks)}")

Total pages: 23
H.M. Samadhi Chathuranga
Rathnayake
IT3011 - Theory & Practices in Statistical Modelling
Linear Regression with Multiple Responses
Total chunks: 24


In [5]:
! pip install langchain-chroma

In [6]:
import google.generativeai as genai
import os

genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

for m in genai.list_models():
    if "embed" in m.name.lower():
        print(m.name, "|", m.supported_generation_methods)

models/gemini-embedding-001 | ['embedContent', 'countTextTokens', 'countTokens', 'asyncBatchEmbedContent']
models/gemini-embedding-2-preview | ['embedContent', 'countTextTokens', 'countTokens', 'asyncBatchEmbedContent']
models/gemini-embedding-2 | ['embedContent', 'countTextTokens', 'countTokens', 'asyncBatchEmbedContent']


In [ ]:
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain.memory import ConversationBufferMemory      # Fix 1: correct import
from langchain_core.prompts import PromptTemplate  
from dotenv import load_dotenv
import os
import shutil

load_dotenv()

# Release the old vectorstore connection first
try:
    vectorstore._client._system.stop()
    del vectorstore
    print("Old vectorstore connection closed.")
except:
    pass  

if os.path.exists("./chroma_db"):
    shutil.rmtree("./chroma_db")
    print("Old vectorstore folder cleared.")

chunks = [c for c in chunks if c.page_content.strip()]
print(f"Chunks being embedded: {len(chunks)}")

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=os.getenv("GEMINI_API_KEY")
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print("Vectorstore created!")

results = vectorstore.similarity_search("what is this document about?", k=3)
for r in results:
    print(r.page_content)
    print("---")

Chunks being embedded: 24
Vectorstore created!
Linear Regression with A Single Response
The equation for a regression model is as follows,
𝑌 = 𝛽 0 + 𝛽1𝑋1 + 𝛽2𝑋2 + 𝛽3𝑋3 + … + 𝛽𝑘𝑋𝑘 + 𝜀
Consider the following population dataset,
Consider that a regression model is going to be fitted to the above dataset with k independent variables to the
response variable Y.
𝑿𝟏 𝑿𝟐 … 𝑿𝒌 𝒀
𝑋11 𝑋12 … 𝑋1𝑘 𝑌1
𝑋21 𝑋22 … 𝑋2𝑘 𝑌2
… … … … …
𝑋𝑛1 𝑋𝑛2 … 𝑋𝑛𝑘 𝑌𝑛
---
Matrix Form of Linear Regression with A Single Response
Consider the following example. Consider these are population data.
What is the design matrix and the response vector.
𝑿 =
1 0
1 1
1
1
1
2
3
4
   𝑌 =
1
4
3
8
9
X 0 1 2 3 4
Y 1 4 3 8 9
---
Linear Regression with A Single Response
The equation for a regression model is as follows,
𝑌 = 𝛽 0 + 𝛽1𝑋1 + 𝛽2𝑋2 + 𝛽3𝑋3 + … + 𝛽𝑘𝑋𝑘 + 𝜀
For each observation in the n observations in the population data,
𝑌1 = 𝛽0 + 𝛽1𝑋11 + 𝛽2𝑋12 + 𝛽3𝑋13 + … + 𝛽𝑘𝑋1𝑘 + 𝜀1
𝑌2 = 𝛽0 + 𝛽1𝑋21 + 𝛽2𝑋22 + 𝛽3𝑋23 + … + 𝛽𝑘𝑋2𝑘 + 𝜀2
𝑌3 = 𝛽0 + 𝛽1𝑋31 + 

In [8]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
import os

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GEMINI_API_KEY")
)

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=os.getenv("GEMINI_API_KEY")
)

# Load existing vectorstore
vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

prompt = ChatPromptTemplate.from_messages([
    ("system", """Answer the question based only on the context below.
    If you don't know, say you don't know.
    
    Context: {context}"""),
    ("human", "{question}")
])

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
)

# Ask questions
question = "What is this document about?"
response = rag_chain.invoke(question)
print(response.content)

This document is about Linear Regression with a Single Response. It describes the general equation for a regression model, how it applies to a population dataset with k independent variables and n observations, and its matrix form, including definitions and examples of the design matrix and response vector.


In [ ]:
print(" Chat with your PDF!")
print("Type 'quit' to exit\n")

while True:
    question = input("You: ")
    if question.lower() in ["quit", "exit"]:
        print("Goodbye!")
        break
    response = rag_chain.invoke({"question": question})
    print(f"\nBot: {response['answer']}")
    print(f"[Retrieved {len(response['source_documents'])} chunks]\n")

 Chat with your PDF!
Type 'quit' to exit

Bot: The document is about **Linear Regression with A Single Response**. It explains the equation for a regression model, how it applies to individual observations in a population dataset with `k` independent variables and a single response variable `Y`, and introduces the **Matrix Form of Linear Regression**, including concepts like the design matrix and response vector.

Bot: The equation for multiple linear regression (with a single response) is:

𝑌 = 𝛽 0 + 𝛽1𝑋1 + 𝛽2𝑋2 + 𝛽3𝑋3 + … + 𝛽𝑘𝑋𝑘 + 𝜀

For the ith observation, it is:

𝑌𝑖 = 𝛽0 + 𝛽1𝑋𝑖1 + 𝛽2𝑋𝑖2 + 𝛽3𝑋𝑖3 + … + 𝛽𝑘𝑋𝑖𝑘 + 𝜀𝑖

Bot: The equation for a linear regression model is:
𝑌 = 𝛽 0 + 𝛽1𝑋1 + 𝛽2𝑋2 + 𝛽3𝑋3 + … + 𝛽𝑘𝑋𝑘 + 𝜀



GoogleGenerativeAIError: Error embedding content (INVALID_ARGUMENT): 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'EmbedContentRequest.content contains an empty Part.', 'status': 'INVALID_ARGUMENT'}}